# RAG from Scratch to Production — Episode 15/20
## The hallucination guard: a second AI call checking the first

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/15_hallucination_guard.ipynb)

After the LLM answers, the [production app](https://github.com/JakkulaMahendhar/rag-chatbot) makes a **second** LLM call: *"Is every claim in this answer supported by the retrieved context?"* If the verdict is "not grounded", it makes a **third** call with a stricter prompt and sends that answer instead.

That doubles the LLM cost of every question. On Episode 2, a reader asked whether calls like this are really needed, and I suggested running the guard only when retrieval looks weak. **This episode tests the guard, and that suggestion.**

**In this episode:**
1. Run the guard on real answers to 24 questions.
2. Feed it 10 subtly made-up answers. How many does it catch?
3. See how different judge models treat "I don't know".
4. Test two ways to call the guard less often.

**Needs an LLM:** Gemini (free key) or local Ollama. About 130 calls.

## 0. Setup

Same models, handbook, search and production prompt as Episode 14.

In [ ]:
%pip install -q google-generativeai requests sentence-transformers

In [1]:
import os, re, json, math, time, requests
import numpy as np

OLLAMA_URL = "http://localhost:11434"
WANTED_OLLAMA = ["llama3.1", "qwen2.5:7b", "gemma2:9b"]   # first available = the main model
GEMINI_MODEL = "gemini-3.6-flash"


def _gemini_key():
    try:
        from google.colab import userdata  # Colab secrets
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.getenv("GEMINI_API_KEY")


MODELS = []
try:
    installed = {m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).json()["models"]}
    MODELS += [m for m in WANTED_OLLAMA if m in installed or f"{m}:latest" in installed]
except Exception:
    pass
if _gemini_key():
    import google.generativeai as genai
    genai.configure(api_key=_gemini_key())
    _gemini = genai.GenerativeModel(GEMINI_MODEL)
    MODELS.append("gemini")

MODEL = MODELS[0]   # answers AND judges, like production
CALLS = {"count": 0}


def generate(prompt: str, model: str = None) -> str:
    model = model or MODEL
    CALLS["count"] += 1
    if model == "gemini":
        time.sleep(6)   # stay under the free tier's rate limit
        return _gemini.generate_content(prompt, generation_config={"temperature": 0}).text.strip()
    data = requests.post(f"{OLLAMA_URL}/api/chat", json={
        "model": model, "messages": [{"role": "user", "content": prompt}], "stream": False,
        "options": {"temperature": 0}}, timeout=600).json()
    return data["message"]["content"].strip()


for m in MODELS:
    generate("Say OK", m)
print("Main model:", MODEL, "| all models:", MODELS)

Main model: llama3.1 | all models: ['llama3.1', 'qwen2.5:7b', 'gemma2:9b']


In [2]:
from sentence_transformers import SentenceTransformer

HANDBOOK = [
    "Remote Work\nEmployees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams. Remote days must be logged in the HR portal by Friday of the previous week.",
    "Annual Leave\nFull-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over into the next year. Book leave with form HR-12.",
    "Parental Leave\nNew parents receive 16 weeks of fully paid leave. Submit form HR-17 at least 8 weeks before the expected start date.",
    "Expenses\nThe meal allowance during business travel is capped at 40 USD per day. Travel expenses are claimed with form EXP-22 within 30 days. Claims above 500 USD need the finance director.",
    "Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses and conferences. It does not roll over. Requests are approved by your direct manager.",
    "Equipment\nEvery employee receives a ThinkPad X1 Carbon laptop, refreshed every 3 years. Lost or stolen devices must be reported to IT within 24 hours.",
    "IT Support\nIf the VPN shows error E-4031, your access token has expired: reset it in the Okta portal. For error E-4032, contact the IT helpdesk.",
    "Security\nMulti-factor authentication is mandatory on all company accounts. Passwords must be at least 14 characters long and must never be reused.",
]
embedder = SentenceTransformer("all-MiniLM-L6-v2")
vectors = embedder.encode(HANDBOOK, normalize_embeddings=True)

def top_chunks(question, k=3):
    return [HANDBOOK[i] for i in np.argsort(-(vectors @ embedder.encode(question, normalize_embeddings=True)))[:k]]

def context(chunks):
    return "\n\n".join(f"SOURCE {i + 1}\n--------------------\nDocument:\nhandbook.pdf\nChunk ID:\n7-{i}\n"
                        f"Similarity Score:\n{0.91 - 0.07 * i:.2f}\nContent:\n{c}" for i, c in enumerate(chunks))

FALLBACK = "I don't have enough information from the provided documents."
RULES = f"""You are an AI assistant specialized in answering questions from provided documents.
Follow these rules strictly:
1. Answer only using the provided context.
2. Do not use external knowledge.
3. If the answer is not available in the context, respond exactly:
"{FALLBACK}"
Do not add additional explanation.
4. Do not create or assume facts.
5. Keep the answer clear and concise.
6. The retrieved context below is labeled with internal metadata - "SOURCE", "Document", "Chunk ID", "Similarity Score" - for your reference only. Never repeat these labels, chunk IDs, filenames, or scores in your answer. Write your answer as if you already knew the information yourself, in natural prose - do not structure it around the source blocks."""

def answer_prompt(question, ctx):
    return f"""{RULES}

Conversation History:
--------------------
No previous conversation.
--------------------
Retrieved Context:
--------------------
{ctx}
--------------------
User Question:
--------------------
{question}
--------------------
Answer:"""

## 1. The guard, exactly as production runs it

Production's guard prompt (`app/services/hallucination_guard_service.py`) and strict re-answer prompt (`rag_chat.py`). Note how the verdict is used:
- Only `grounded` (true/false) matters. The `confidence` it asks for is never read, and the `hallucination_threshold` setting (0.75) isn't used anywhere.
- If the reply can't be parsed, it counts as **not grounded**.
- The strict re-answer is **not** checked again.

In [3]:
GUARD = """
You are a hallucination detection system.
Your task:
Check whether the answer is fully supported by the provided context.
Rules:
- Do not use external knowledge.
- Identify unsupported claims.
- Return only JSON.
Question:
{question}
Context:
{context}
Answer:
{answer}
Return:
{{
    "grounded": true/false,
    "confidence": 0-1,
    "unsupported_claims": []
}}
"""

STRICT = """
You are a strict RAG assistant.
Answer ONLY from the context.
If information is missing say:
"I don't have enough information."
The context below is labeled with internal metadata ("SOURCE",
"Document", "Chunk ID", "Similarity Score") for your reference only -
never repeat these labels, chunk IDs, filenames, or scores in your
answer. Write natural prose, not a source-by-source breakdown.
Context:
{context}
Question:
{question}
"""


def grounded(question, ctx, answer, model=None):
    """Production's verdict: True / False. Unparseable counts as False."""
    reply = generate(GUARD.format(question=question, context=ctx, answer=answer), model).strip()
    if reply.startswith("```"):
        reply = reply.replace("```json", "").replace("```", "").strip()
    try:
        return bool(json.loads(reply)["grounded"])
    except Exception:
        return False


def production_pipeline(question):
    ctx = context(top_chunks(question))
    first = generate(answer_prompt(question, ctx))
    ok = grounded(question, ctx, first)
    final = first if ok else generate(STRICT.format(context=ctx, question=question))
    return first, ok, final

## 2. Test 1: real answers

24 questions: 14 the handbook answers (with the words a correct answer contains), plus 10 it can't answer, where the right reply is a refusal.

In [4]:
ANSWERABLE = [
    ("How many weeks of leave do new parents get?",          ["16"]),
    ("How many vacation days carry over to next year?",      ["5"]),
    ("Does the learning budget carry over to next year?",    ["not", "no"]),
    ("Who has to approve expense claims over 500 USD?",      ["finance director"]),
    ("Who approves my course and conference requests?",      ["manager"]),
    ("What happens if my VPN access token expired?",         ["okta", "reset"]),
    ("How fast do I report a stolen phone?",                 ["24 hours"]),
    ("Which day must I be in the office?",                   ["wednesday"]),
    ("Can I reuse an old password?",                         ["never", "not", "no"]),
    ("What's the deadline for logging remote days?",         ["friday"]),
    ("How long do I have to submit a travel expense claim?", ["30 days"]),
    ("When must the parental leave form be submitted?",      ["8 weeks"]),
    ("Can I save unused training money for next year?",      ["not", "no"]),
    ("What do I get for food when travelling for work?",     ["40"]),
]
UNANSWERABLE = [
    "How many sick days do I get per year?", "How much is the gym allowance?", "Can I bring my dog to the office?",
    "Is there a company pension plan?", "What is the bonus for referring a new hire?", "How much is the parking allowance?",
    "What is the capital of France?", "Who won the football World Cup in 2022?",
    "What is the dress code for the office party?", "How do I bake sourdough bread?",
]

def is_refusal(text):
    return "enough information" in text.lower()

def is_correct(text, words):
    return is_refusal(text) if words is None else (not is_refusal(text) and any(w in text.lower() for w in words))


CALLS["count"] = 0
RESULTS = []
for question, words in ANSWERABLE + [(q, None) for q in UNANSWERABLE]:
    first, ok, final = production_pipeline(question)
    RESULTS.append((question, words, first, ok, final))
    if not ok or not is_correct(first, words):
        print(f"{'✅' if is_correct(first, words) else '❌'} first answer   guard: {'grounded' if ok else 'FLAGGED'}   {question}")
        print(f"      first: {first[:70]!r}")
        if not ok:
            print(f"      final: {final[:70]!r}")

print(f"\nFirst answers correct: {sum(is_correct(f, w) for _, w, f, _, _ in RESULTS)}/24")
print(f"Guard flagged:         {sum(not ok for *_, ok, _ in RESULTS)}/24")
print(f"LLM calls:             {CALLS['count']} (without the guard: 24)")

❌ first answer   guard: grounded   How fast do I report a stolen phone?
      first: "I don't have enough information from the provided documents."


❌ first answer   guard: grounded   Can I save unused training money for next year?
      first: "I don't have enough information from the provided documents."


First answers correct: 22/24
Guard flagged:         0/24
LLM calls:             48 (without the guard: 24)


In my local run (llama3.1), the guard flagged **nothing**: every real answer was judged grounded. The LLM calls doubled, and nothing changed.

But look at the two wrong first answers: both are **refusals** of questions the handbook does answer (the stolen phone, the training money). The guard approved them. That's by design: it checks whether what the answer **says** is supported, not whether the answer is **complete**. A refusal says nothing false, so it passes.

**The guard catches made-up claims, not missing answers.**

## 3. Test 2: planted hallucinations

Real answers rarely hallucinated here, so let's write some. For 10 questions: the true answer, and a subtly made-up one (a wrong number, a wrong approver, an invented extra rule).

In [5]:
PLANTED = [
    ("How many weeks of leave do new parents get?", "New parents receive 16 weeks of fully paid leave.", "New parents receive 20 weeks of fully paid leave."),
    ("Who has to approve expense claims over 500 USD?", "The finance director must approve them.", "Your direct manager must approve them."),
    ("How fast do I report a stolen phone?", "Within 24 hours, to IT.", "Within 48 hours, to IT."),
    ("Which day must I be in the office?", "Wednesday is the mandatory in-office day.", "Monday is the mandatory in-office day."),
    ("Can I reuse an old password?", "No, passwords must never be reused.", "Yes, after 12 months you may reuse an old password."),
    ("What's the deadline for logging remote days?", "By Friday of the previous week.", "By Friday of the previous week, and your manager must approve each day in advance."),
    ("How long do I have to submit a travel expense claim?", "Within 30 days, with a receipt.", "Within 30 days; receipts are only needed above 100 USD."),
    ("Does the learning budget carry over to next year?", "No, it does not roll over.", "Yes, up to 300 USD can be carried over."),
    ("What happens if my VPN access token expired?", "You see error E-4031; reset the token in the Okta portal.", "You see error E-4032; contact the IT helpdesk to reset it."),
    ("How many vacation days carry over to next year?", "Up to 5 unused days.", "Up to 10 unused days."),
]

caught = false_alarms = 0
for question, true_answer, fake_answer in PLANTED:
    ctx = context(top_chunks(question))
    fake_ok = grounded(question, ctx, fake_answer)
    true_ok = grounded(question, ctx, true_answer)
    caught += not fake_ok
    false_alarms += not true_ok
    if fake_ok:
        print(f"   missed: {fake_answer!r}")
    if not true_ok:
        print(f"   false alarm on: {true_answer!r}")
print(f"Fakes caught: {caught}/10   False alarms on true answers: {false_alarms}/10")

   missed: 'Within 30 days; receipts are only needed above 100 USD.'


Fakes caught: 9/10   False alarms on true answers: 0/10


As a fake-detector, the guard works well: it caught nearly every planted hallucination without flagging the true answers. When the answering model **does** invent something, the guard is a real safety net.

The one that got through in my run is worth noticing: *"receipts are only needed above 100 USD"*, an **invented exception** added to a true rule. Subtle additions are the hardest to catch.

## 4. Test 3: what does the judge make of "I don't know"?

The refusal sentence is a special case. A refusal can be **correct** (the answer isn't in the documents) or **wrong** (it is). Each available model judges both kinds.

*A correction to Episode 14:* there I only tested the refusal on questions the handbook **does** answer, where refusing is the wrong answer, and described the extra call as happening "because the bot correctly said I don't know". The clean test below separates the two cases.

In [6]:
WRONG_REFUSALS = ["How long must passwords be?", "Who approves claims above 500 USD?",
                  "How many weeks of parental leave?", "What does VPN error E-4031 mean?"]
CORRECT_REFUSALS = ["How many sick days do I get per year?", "How much is the gym allowance?",
                    "Is there a company pension plan?", "What is the capital of France?"]

for m in MODELS:
    wrong = sum(grounded(q, context(top_chunks(q)), FALLBACK, m) for q in WRONG_REFUSALS)
    right = sum(grounded(q, context(top_chunks(q)), FALLBACK, m) for q in CORRECT_REFUSALS)
    print(f"{m:<12} approves wrong refusals {wrong}/4   approves correct refusals {right}/4")

llama3.1     approves wrong refusals 3/4   approves correct refusals 4/4


qwen2.5:7b   approves wrong refusals 0/4   approves correct refusals 0/4


gemma2:9b    approves wrong refusals 1/4   approves correct refusals 0/4


In my run, the judges split into two camps, and **neither is right**:
- **Approves (almost) every refusal** (llama3.1, production's default): correct refusals pass, but so do wrong ones. No extra calls, but no protection against missing answers either.
- **Rejects (almost) every refusal** (qwen2.5, gemma2): wrong refusals get a second chance, but so does every **correct** "I don't know", at the cost of a third LLM call each time.

None of them tells the two apart, because the guard's question ("is every claim supported?") doesn't fit a refusal, which makes no claims.

## 5. Calling the guard less often

**Idea A, from my Episode 2 reply: only run the guard when retrieval is weak** (the reranker's best score below 0.20, from Episode 11). The logic: weak retrieval → more risk of a made-up answer.

In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def best_rerank_score(question):
    scores = reranker.predict([[question, c] for c in top_chunks(question)])
    return 1 / (1 + math.exp(-float(max(scores))))

all_questions = [q for q, *_ in RESULTS]
weak = [q for q in all_questions if best_rerank_score(q) < 0.20]
fakes_checked = sum(best_rerank_score(q) < 0.20 for q, *_ in PLANTED)
print(f"Guard runs on {len(weak)}/24 real questions (saves {24 - len(weak)} calls)")
print(f"But only {fakes_checked}/10 planted fakes would be checked at all")

Guard runs on 13/24 real questions (saves 11 calls)
But only 1/10 planted fakes would be checked at all


It saves calls, but it **misses most fakes**: the planted hallucinations are answers to perfectly clear questions, where retrieval was strong. A model can invent a wrong number even when the right chunk is sitting in front of it. **Hallucination risk isn't the same as retrieval weakness.** My suggestion doesn't hold up.

**Idea B: skip the guard when the answer is the exact refusal sentence.** A refusal makes no claims, so there's nothing for the guard to check. And since production's prompt makes refusals word-for-word identical (Episode 13), they're easy to detect.

In [8]:
refusals = sum(is_refusal(first) for _, _, first, _, _ in RESULTS)
print(f"Guard calls saved: {refusals}/24 (every refusal)")
print(f"Planted fakes still checked: 10/10 (none of them is a refusal)")

Guard calls saved: 12/24 (every refusal)
Planted fakes still checked: 10/10 (none of them is a refusal)


About half the guard calls saved, no fakes missed, and for judges that reject every refusal, no more wasted re-answers.

The remaining weakness, **wrong refusals**, needs a different check: Episode 11's reranker bands already flag "the documents look relevant, but the bot refused". That's a much cheaper signal than another LLM call.

## 6. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot):
- Runs the guard on **every** answer, including refusals: 2 LLM calls per question (3 when flagged).
- Uses only the true/false verdict. `confidence` and the `hallucination_threshold` setting (0.75) are unused.
- Uses the **same model** to answer and to judge. With llama3.1 (the default) refusals are always approved; switch providers and every correct refusal may trigger a re-answer.
- Doesn't re-check the strict re-answer.

## Recap

| Question | What we measured |
|---|---|
| Does the guard catch made-up claims? | Yes: nearly all planted fakes, no false alarms |
| Did real answers need it? | Not here: 0 of 24 flagged, while LLM calls doubled |
| Does it catch wrong refusals? | No. A refusal makes no claims, so it can't be "unsupported" |
| "Only guard when retrieval is weak"? | Checked only a fraction of the fakes. **Rejected** |
| "Skip the guard on refusals"? | About half the guard calls saved, no fakes missed |

**The rule:** a second LLM call is worth it only for what it can actually catch. Point it at claims, not at refusals, and use cheaper signals for the rest.

**Next, Episode 16: evaluating RAG.** Everything this series measured, turned into one repeatable test suite, including the known-facts idea from a reader on Episode 3.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)